# 01 — Internal research, training, and comparison

This is the single runnable narrative for the historical internal Defactify study: environment checks, dataset audit, controlled rasterisation, FFT exploration, and read-only inspection of observed artifacts. Run sections in order. No cell invents metrics when an artifact is absent or restarts the intentionally stopped H1-N neural series.

**Scope:** historical exploratory internal evidence only. HighRes-v1 remains at source-audit stage; the confirmatory external and robustness phase in `02_external_validation_and_results.ipynb` is therefore also locked.

## 1. Environment and reproducibility


Run this notebook before a controlled experiment. It records the software environment and checks the locked H1-N preprocessing contract. It contains no model-accuracy claim.

**Status discipline.** D0 denotes the completed legacy diagnostic controls that exposed a geometry/source confound. H1-N denotes the amended, source-normalised comparison. D0 metrics are not H1-N results, and no completed H1-N neural run is assumed by this notebook. A model is not selected for an interface until the separately locked external evaluation is complete.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    CONTROLLED_PREPROCESSING_PROTOCOL,
    preprocessing_metadata,
)
from ai_image_detector.reproducibility import (
    environment_snapshot,
    get_device,
    save_json,
    seed_everything,
)


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'ai_image_detector').is_dir():
            return candidate
    raise RuntimeError('Open Jupyter from this repository or one of its subdirectories.')


REPO = find_repository_root()
SEED = 7
seed_everything(SEED)
DEVICE = get_device()
H1N_PREPROCESSING = preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)

assert H1N_PREPROCESSING['image_size'] == CONTROLLED_IMAGE_SIZE == 128
assert H1N_PREPROCESSING['train_crop'] == 'seeded_random_square_crop'
assert H1N_PREPROCESSING['eval_crop'] == 'center_square_crop'
assert H1N_PREPROCESSING['neural_train_augmentation']['horizontal_flip']['probability'] == 0.5

snapshot = environment_snapshot() | {
    'seed': SEED,
    'selected_device': str(DEVICE),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'h1n_preprocessing': H1N_PREPROCESSING,
}
save_json(snapshot, REPO / 'artifacts/environment/environment.json')
print(f'Repository: {REPO}')
snapshot

In [ ]:
artifact_root = REPO / 'artifacts'
completed_h1n = []
for run_path in artifact_root.glob('*/run.json'):
    run = json.loads(run_path.read_text(encoding='utf-8'))
    protocol = run.get('preprocessing', {}).get('protocol')
    metrics_path = run_path.parent / 'internal_test_metrics.json'
    if protocol == CONTROLLED_PREPROCESSING_PROTOCOL and metrics_path.is_file():
        completed_h1n.append(run_path.parent.name)

study_status = {
    'D0_legacy_diagnostics': 'completed; retained only as a confound diagnostic',
    'H1_N_completed_internal_runs': sorted(completed_h1n),
    'H1_N_confirmatory_external_evaluation': 'locked and pending',
    'deployable_model': 'none until the frozen external evaluation is reported',
}
study_status

## Acceptance checks

A controlled run may proceed only when the device and Git revision are recorded, the H1-N metadata says `h1n_square_crop_128_v1` and `128 × 128`, and the grouped-manifest gate in Notebook 01 passes. For neural H1-N RGB/FFT training, the train crop is a seeded random square crop; neural validation and evaluation use the deterministic centre crop. The controlled radial-logistic baseline is intentionally different: it extracts deterministic centre-crop features on train, validation, and test, with no augmentation. On Apple Silicon, record the MPS availability; do not treat CPU/MPS choice as a performance result.

In [ ]:
assert torch.__version__, 'PyTorch is unavailable'
print(json.dumps(snapshot, indent=2))
print('MPS available:', torch.backends.mps.is_available())
print('No accuracy, calibration, or deployment claim is produced in this notebook.')

## 2. Data audit and leakage-resistant split


This notebook audits the grouped Defactify manifest before H1-N training. The official split had cross-split caption and perceptual-hash overlap, so the group-disjoint manifest is the internal experimental frame. The original internal test was inspected during D0; therefore any H1-N result on it is an **exploratory internal stress-test result**, not the confirmatory result.

In [ ]:
from pathlib import Path

import pandas as pd

from ai_image_detector.manifest import audit_summary, load_manifest, split_overlap_report

MANIFEST = REPO / 'data/processed/defactify_grouped/manifest.csv'
assert MANIFEST.exists(), 'Run prepare_defactify.py and make_grouped_split.py first.'
frame = load_manifest(MANIFEST, check_paths=True)
required_columns = {'label', 'split', 'generator', 'width', 'height', 'leakage_group', 'group_id', 'source_id', 'sha256', 'phash'}
missing_columns = required_columns.difference(frame.columns)
assert not missing_columns, f'Manifest misses controlled-protocol columns: {sorted(missing_columns)}'
frame.head()

In [ ]:
summary = audit_summary(frame)
for name, value in summary.items():
    print(f'\n--- {name} ---')
    display(value) if hasattr(value, 'style') else print(value)

for key in ('source_id', 'group_id', 'leakage_group', 'caption', 'sha256', 'phash'):
    if key in frame.columns:
        leaked = split_overlap_report(frame, key)
        print(f'{key}: {len(leaked)} records in a cross-split group')
        if len(leaked):
            display(leaked.head())

## Why D0 required an amendment

The prepared corpus has a class-correlated geometry/source channel: real photographs have varied rectangular dimensions, whereas synthetic images are square, and no exact `(width, height)` pair is shared across the two labels. D0's metadata-only control and direct rectangular-to-square resizing therefore showed that geometry can produce a high score without demonstrating image provenance. In particular, anisotropic resizing can create a class-correlated frequency pattern before an FFT is calculated.

D0 remains a useful *diagnostic* record. It is not an H1-N baseline, an architecture-selection result, or evidence that an individual image is AI-generated.

In [ ]:
geometry = (
    frame.assign(aspect_ratio=frame['width'] / frame['height'])
    .groupby(['label', 'generator'], dropna=False)
    .agg(
        images=('path', 'size'),
        unique_widths=('width', 'nunique'),
        unique_heights=('height', 'nunique'),
        median_aspect_ratio=('aspect_ratio', 'median'),
    )
    .reset_index()
)

real_dimensions = set(map(tuple, frame.loc[frame.label == 0, ['width', 'height']].to_numpy()))
fake_dimensions = set(map(tuple, frame.loc[frame.label == 1, ['width', 'height']].to_numpy()))
geometry_gate = {
    'exact_width_height_pairs_shared_between_labels': len(real_dimensions & fake_dimensions),
    'real_images_square_fraction': float((frame.loc[frame.label == 0, 'width'] == frame.loc[frame.label == 0, 'height']).mean()),
    'fake_images_square_fraction': float((frame.loc[frame.label == 1, 'width'] == frame.loc[frame.label == 1, 'height']).mean()),
}
display(geometry)
geometry_gate

## H1-N paired group sampler

The neural H1-N RGB/FFT runs train from the group-disjoint manifest with `paired_group_balanced_v1`. Each group visit emits one real image and one fake sibling; each neural training image then receives its seeded random square crop and train-only horizontal flip. The crop/flip RNG is keyed by seed, epoch, and stable `source_id`, not the absolute checkout path. The fake generator is assigned from a balanced, seeded cycle, so large duplicate groups and a more frequent generator cannot obtain extra training weight. Neural validation/test use the deterministic centre crop. The controlled radial-logistic baseline does not use this sampler or random crop: it extracts one deterministic centre-crop feature vector per image on every split. This is a training balance mechanism; it does not make the internal test confirmatory.

In [ ]:
from itertools import islice

from ai_image_detector.training import (
    PAIRED_GROUP_BALANCED_SAMPLER,
    PairedGroupSampler,
)

train_frame = frame.loc[frame.split == 'train'].reset_index(drop=True)
sampler = PairedGroupSampler(train_frame, seed=7, group_column='leakage_group')
sampled_indices = list(islice(iter(sampler), 12))
sampled = train_frame.iloc[sampled_indices].copy()

for start in range(0, len(sampled_indices), 2):
    pair = sampled.iloc[start : start + 2]
    assert pair.label.tolist() == [0, 1]
    assert pair.leakage_group.nunique() == 1

display(sampled[['leakage_group', 'label', 'generator', 'path']])
assert sampler.metadata()['choice'] == PAIRED_GROUP_BALANCED_SAMPLER
sampler.metadata()

## Decision gate

Proceed only if all group/caption/near-duplicate checks are clean for the grouped manifest and the sampler check passes. H1-N then uses a source-normalised 128 × 128 raster for **both** representations; Section 3 of this notebook makes that contract visible. Do not use accuracy alone, D0 metrics, or a smoke run to choose a deployment model.

## 3. FFT representation exploration


This is descriptive analysis, not provenance proof. It visualises the exact H1-N pixel contract: decode to RGB, crop a source-coordinate square without padding, resize once to 128 × 128, then pass the same raster either to RGB or to the FFT-magnitude transform. It does **not** reproduce the legacy D0 direct-resize radial result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    fft_magnitude,
    radial_power_spectrum,
    source_normalized_rasterize,
)
from ai_image_detector.manifest import load_manifest

frame = load_manifest(REPO / 'data/processed/defactify_grouped/manifest.csv', check_paths=True)
train = frame.loc[frame.split == 'train'].copy()

for group, candidate in train.groupby('leakage_group', sort=True):
    if {0, 1}.issubset(set(candidate.label)):
        chosen_group = group
        break
else:
    raise RuntimeError('No real/fake paired group was found in the grouped train split.')

examples = (
    train.loc[train.leakage_group == chosen_group]
    .sort_values(['label', 'generator'])
    .groupby('generator', as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
examples[['leakage_group', 'label', 'generator', 'width', 'height', 'path']]

In [ ]:
fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
axes = np.atleast_2d(axes)

for row_axes, (_, row) in zip(axes, examples.iterrows(), strict=True):
    original = Image.open(row.path).convert('RGB')
    raster = source_normalized_rasterize(original, size=CONTROLLED_IMAGE_SIZE, train=False)
    magnitude = fft_magnitude(raster, size=CONTROLLED_IMAGE_SIZE)
    radial = radial_power_spectrum(raster, size=CONTROLLED_IMAGE_SIZE)

    assert raster.size == (CONTROLLED_IMAGE_SIZE, CONTROLLED_IMAGE_SIZE)
    row_axes[0].imshow(original)
    row_axes[0].set_title(f'original: {row.width}×{row.height}')
    row_axes[1].imshow(raster)
    row_axes[1].set_title('H1-N centre-crop then 128×128')
    row_axes[2].imshow(magnitude, cmap='magma')
    row_axes[2].set_title('FFT magnitude of common raster')
    row_axes[3].plot(radial)
    row_axes[3].set_title('radial spectrum of common raster')
    for axis in row_axes[:3]:
        axis.set_axis_off()

plt.tight_layout()

## What is controlled, and what is not

Neural H1-N RGB/FFT training uses a seeded random square crop; neural validation, exploratory internal test, robustness, and external evaluation use the deterministic centre crop shown above. The observed controlled radial-logistic baseline deliberately avoids stochastic feature extraction: it uses the deterministic centre crop on train, validation, and test, without augmentation. The radial curve in this static visualisation likewise comes from the deterministic evaluation raster. Letterboxing and direct rectangular-to-square resizing are prohibited because they expose geometry or interpolation as a possible label cue.

Visible spectral patterns remain hypothesis-generating. H1-N asks whether an FFT-magnitude ResNet-50 outperforms an equal-capacity RGB ResNet-50 under this shared rasterisation, across three predeclared seeds. A plot, a legacy radial score, or an individual softmax output does not establish an image's origin.

## 4. Historical H1-N record and current training gate


H1-N was a narrow intended representation comparison: FFT-magnitude ResNet-50 versus RGB ResNet-50, both trained from random initialisation on the same source-normalised 128 × 128 raster and paired-group training stream. The only completed neural run is one RGB pilot. The controlled radial-logistic baseline is a separate deterministic feature baseline. The original direct-resize experiments are D0 diagnostics and are deliberately excluded from this comparison.

The planned six-run H1-N series was intentionally stopped after the pilot and must not be resumed opportunistically. This notebook keeps the protocol and observed artifacts readable, but exposes no executable H1-N neural-training, analysis, or aggregation command. The next training cell belongs to a separately frozen HighRes-v1 corpus after its source and leakage audit pass.

In [ ]:
import json

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL, preprocessing_metadata
from ai_image_detector.manifest import load_manifest

MANIFEST = REPO / 'data/processed/defactify_grouped/manifest.csv'
assert MANIFEST.is_file(), 'Run the grouped-split preparation before H1-N training.'
frame = load_manifest(MANIFEST, check_paths=True)
assert {'train', 'val', 'test'}.issubset(set(frame.split))
assert preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)['image_size'] == 128

print(frame.groupby(['split', 'label']).size().rename('n'))

## D0 record: read-only diagnostic evidence

The old radial-FFT and file-metadata artifacts may be inspected to document why the protocol changed. They must not enter a H1-N ranking, threshold choice, model card, or web interface. The observed controlled radial-logistic baseline is distinct from this legacy D0 radial artifact: it uses a deterministic H1-N centre crop for train, validation, and test and supplies one fixed pixel-only feature vector per image. The metadata control receives geometry/source information unavailable to the intended image model; its score is evidence of dataset bias, not detection quality.

In [ ]:
print('D0 artifacts are historical diagnostics only. Consult the evidence-bound coursework draft for their recorded metrics.')

## Controlled radial baseline: recorded evidence

The observed radial logistic baseline is a deterministic fixed-feature control, not a neural run. It is retained as a recorded result in the report and is not rerun from this notebook, because the historical internal test has already been inspected and the project has moved to a separately controlled high-resolution source decision.

In [ ]:
print('Historical radial baseline: ROC-AUC 0.708864, BAcc 0.653833, macro-F1 0.587577.')
print('Status: exploratory internal evidence; not eligible for selection, external evaluation, or the interface.')

## H1-N neural series: intentionally stopped

The predeclared RGB/FFT × three-seed H1-N series is not completed and will not be resumed. Its original equal-budget rules remain documented in the research protocol so that the stopping decision is auditable. The single historical RGB run is a pilot, not a selected model or a basis for a web interface.

In [ ]:
H1N_NEURAL_STATUS = {
    'status': 'intentionally_stopped',
    'completed_neural_runs': 1,
    'remaining_runs_launched': 0,
    'reason': 'A new high-resolution source and leakage protocol is required before further model training.',
}
H1N_NEURAL_STATUS

## Future HighRes-v1 analysis gate

A future HighRes-v1 comparison will use a frozen corpus, declared architecture/representation/seed rules, and group-aware uncertainty before the internal test is read. It will report per-source and per-generator slices without selecting a favourable seed. This work cannot begin until a source passes the catalog, provenance, image-byte, and leakage gates.

In [ ]:
print('PENDING: no HighRes-v1 analysis command exists until a primary corpus is frozen.')

## Future aggregation gate

Aggregation will be exposed only with a future HighRes-v1 protocol that declares its seed set before training. The stopped H1-N series is not backfilled or aggregated as if it were complete.

In [ ]:
print('PENDING: aggregation is disabled until a completed pre-registered HighRes-v1 series exists.')

## Selection and external-evaluation gate

A future HighRes-v1 result must report ROC-AUC, PR-AUC, balanced accuracy, macro-F1, class recalls, FPR at TPR 95%, per-generator values, calibration, and group-aware uncertainty. Threshold selection remains validation-only. The locked Synthbuster + RAISE evaluation opens only after the HighRes-v1 model family, preprocessing, checkpoint rule, and threshold rule are frozen. No historical H1-N artifact is eligible for this selection path.

## DANI HighRes-v1: completed data gate

The active primary study uses a frozen 7,410-row DANI selection grouped by 1,482 COCO parents. The original encoded files are retained for provenance and a metadata-shortcut control. Neural models use the derived 1024 × 1024 RGB PNG corpus, which is downsampled exactly once to the common 384 × 384 model raster. This cell fails closed unless the repeated integrity audit permits training.

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DANI_AUDIT = PROJECT_ROOT / 'artifacts/audits/dani_rgb1024_integrity_v1/summary.json'
dani_audit = json.loads(DANI_AUDIT.read_text(encoding='utf-8'))
assert dani_audit['eligibility']['eligible_for_training'] is True
assert dani_audit['counts']['cross_label_exact_duplicate_group_count'] == 0
assert dani_audit['counts']['cross_split_integrity_component_count'] == 0
assert dani_audit['shortcut_audit']['format_mode_support_balanced_between_labels'] is True
dani_audit['counts']

## Acquisition-pipeline shortcut control

The metadata-only logistic control is deliberately evaluated on validation while internal test remains closed. A strong score on original files is evidence that the source containers are informative, not evidence of synthesis detection. The comparison with the canonical corpus estimates how much of that simple association remains after normalising container and colour mode.

In [ ]:
shortcut_paths = {
    'original_source_files': PROJECT_ROOT / 'artifacts/baselines/dani_source_shortcut_validation_v1/file_metadata_control_seed7/validation_selection_metrics.json',
    'canonical_rgb_png': PROJECT_ROOT / 'artifacts/baselines/dani_canonical_metadata_validation_v1/file_metadata_control_seed7/validation_selection_metrics.json',
}
shortcut_results = {name: json.loads(path.read_text(encoding='utf-8')) for name, path in shortcut_paths.items()}
pd.DataFrame.from_dict(shortcut_results, orient='index')[['roc_auc', 'balanced_accuracy', 'macro_f1', 'ece_15']]

## Validation-only MPS training series

The one-epoch seed-7 MPS run is a runtime pilot and is excluded from the research table. The predeclared practical series consists of ImageNet-pretrained RGB ResNet-50 runs for seeds 7, 17 and 42, with batch 64, learning rate 1e-4, at most 15 epochs, patience 4 and paired parent sampling. This notebook reads only validation artifacts; the absence of `internal_test_metrics.json` is an asserted part of the model-selection protocol.

In [ ]:
experiment_root = PROJECT_ROOT / 'artifacts/experiments'
validation_rows = []
for seed in (7, 17, 42):
    run_dir = experiment_root / f'dani_rgb_pretrained_seed{seed}_validation_v1'
    metrics_path = run_dir / 'validation_metrics.json'
    if metrics_path.exists():
        assert not (run_dir / 'internal_test_metrics.json').exists()
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        validation_rows.append({'seed': seed, **metrics})
validation_table = pd.DataFrame(validation_rows)
validation_table

### Test lock

Do not evaluate internal test from this notebook until all declared validation runs, the cross-seed aggregation, and the representative-checkpoint rule are complete. A pilot or a favourable validation seed cannot open the test set. External evaluation remains a later, separately frozen stage.